# Figure S2C — 1-Year Cumulative Incidence by Sex

Self-contained notebook. KM-based 1-year cumulative incidence per toxicity, stratified by sex,
with 95% CI error bars. Male vs. Female is additionally compared per toxicity with a **multivariable**
Cox proportional-hazards model (Male = reference, adjusted for age and cancer type), and the
resulting HR / p-value are annotated on the plot and included in the CSV.

**Cohort:** first line of therapy only. `line1_start` and censoring are derived from the *first*
LOT row per patient.

**Censoring (from `line1`):** `min(lot_end + 180d, next_lot_start, death, last_fu)`, measured from `line1_start`.

**Outputs:**
- `Sex_Prevalance_S2C.pdf` — the figure
- `Sex_Prevalance_S2C_cumulative_incidence.csv` — 1-year CI / 95% CI per sex × toxicity, plus
  the Cox PH hazard ratio and p-value for Female vs. Male on each toxicity row

**Formatting (Nature compliance):**
- Arial only (no fallback substitution), hard-fails if not resolved
- `pdf.fonttype = 42` / `ps.fonttype = 42` (text stays editable, not outlined)
- Tiered text sizes: 7pt axis labels, 6pt tick labels, 5pt legend
- No `bbox_inches='tight'` on save — figure is placed in Illustrator at 100% scale, unresized


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import KaplanMeierFitter

%matplotlib inline

warnings.filterwarnings('ignore')

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

## Paths

Notebook lives in `figure 2/scripts/`. Data lives in the sibling `figure 2/data/` folder; outputs go to
`figure 2/results/supp/S2C_Sex_Prevalence/` (rename below if a different folder name is preferred).


In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S2C_Sex_Prevalence'))
os.makedirs(RESULTS_DIR, exist_ok=True)

LLM_PATIENT_PATH = os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv')
LLM_BATCH_PATH = os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv')

PDF_OUT = os.path.join(RESULTS_DIR, 'Sex_Prevalance_S2C.pdf')
CSV_OUT = os.path.join(RESULTS_DIR, 'Sex_Prevalance_S2C_cumulative_incidence.csv')

for p in [LLM_PATIENT_PATH, LLM_BATCH_PATH]:
    print(('FOUND   ' if os.path.exists(p) else 'MISSING '), p)

## Constants


In [ ]:
TOXICITY_COLUMNS = ['liver_toxicity', 'hypothyroidism', 'pneumonitis',
                    'colitis', 'adrenal_insufficiency', 'hyperthyroidism']

TOXICITY_DISPLAY = {
    'pneumonitis': 'Pneumonitis', 'adrenal_insufficiency': 'Adrenal Insufficiency',
    'liver_toxicity': 'Liver Toxicity', 'colitis': 'Colitis',
    'hyperthyroidism': 'Hyperthyroidism', 'hypothyroidism': 'Hypothyroidism',
}

SEX_GROUPS = ['Male', 'Female']
SEX_GROUP_COLORS = {'Male': '#3498DB', 'Female': '#E74C3C'}

T_MONTHS = 12.0

import re

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r'\d+', str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None


## Build `line1` (first-line censoring) and `patient_covars` (first LOT per patient)


In [ ]:
COVARS_PATH = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026', 'llm84k_pneumonitis_grade0_20260630.csv')
covars = pd.read_csv(COVARS_PATH, low_memory=False)
covars['mrn'] = covars['mrn'].apply(standardize_mrn)
covars = covars[covars['mrn'].notna()].copy()

covars['lot'] = pd.to_numeric(covars['lot'], errors='coerce')
covars['lot_start'] = pd.to_datetime(covars['lot_start'], errors='coerce')
covars['censor_days'] = pd.to_numeric(covars['t_cutoff_lot'], errors='coerce')
covars = covars.sort_values(['mrn', 'lot'])

# ---- line1: first LOT per patient, with pre-computed censoring ----
covars_valid = covars[covars['lot_start'].notna()].copy()
idx = covars_valid.sort_values(['mrn', 'lot']).groupby('mrn')['lot'].idxmin()
line1 = (covars_valid.loc[idx, ['mrn', 'lot_start', 'censor_days']]
         .rename(columns={'lot_start': 'line1_start'}))
line1 = line1[np.isfinite(line1['censor_days']) & (line1['censor_days'] > 0)].copy()
print(f'{len(line1):,} patients with line 1 dates and valid censoring')

# ---- patient_covars: first LOT per patient ----
idx2 = covars.groupby('mrn')['lot'].idxmin()
patient_covars = covars.loc[idx2].reset_index(drop=True)
print(f'{len(patient_covars):,} patients with covariates')

## Load LLM patient-level cohort and build sex groups

`llm_merged` restricts the cohort to patients who have LLM predictions. Sex values are cleaned
(stripped, capitalized) and restricted to `Male`/`Female`.


In [ ]:
llm_patients = pd.read_csv(LLM_PATIENT_PATH, encoding='latin-1', low_memory=False)
llm_patients['mrn'] = llm_patients['mrn'].apply(standardize_mrn)
llm_patients = llm_patients[llm_patients['mrn'].notna()].copy()
for tox in TOXICITY_COLUMNS:
    if tox in llm_patients.columns:
        llm_patients[tox] = pd.to_numeric(llm_patients[tox], errors='coerce').fillna(0).astype(int)
print(f'{len(llm_patients):,} patients with LLM predictions')

llm_merged = llm_patients.merge(patient_covars, on='mrn', how='inner', suffixes=('', '_cov'))
print(f'{len(llm_merged):,} patients after merge (LLM predictions ∩ covariates)')

sex_df = llm_merged[['mrn', 'sex']].drop_duplicates('mrn').copy()
sex_df['sex_clean'] = sex_df['sex'].astype(str).str.strip().str.capitalize()
sex_df = sex_df[sex_df['sex_clean'].isin(SEX_GROUPS)].copy()
print(sex_df['sex_clean'].value_counts().reindex(SEX_GROUPS))


## Load batch-level toxicity data and merge with `line1`


In [ ]:
batch_df = pd.read_csv(LLM_BATCH_PATH, encoding='latin-1', low_memory=False)
batch_df['mrn'] = batch_df['mrn'].apply(standardize_mrn)
batch_df = batch_df[batch_df['mrn'].notna()].copy()
batch_df['window_start'] = pd.to_datetime(batch_df['window_start'], errors='coerce')
batch_df['window_end'] = pd.to_datetime(batch_df['window_end'], errors='coerce')
batch_df = batch_df.dropna(subset=['window_start', 'window_end'])
print(f'{len(batch_df):,} batch records for {batch_df["mrn"].nunique():,} patients')

batch_merged = batch_df.merge(line1, on='mrn', how='inner')
batch_merged['days_from_start'] = (batch_merged['window_start'] - batch_merged['line1_start']).dt.days
batch_merged = batch_merged[(batch_merged['days_from_start'] >= 0) &
                            (batch_merged['days_from_start'] <= batch_merged['censor_days'])].copy()
print(f'{len(batch_merged):,} batch records within the censoring window')


## KM 1-year cumulative incidence with 95% CI

For a given set of MRNs and a toxicity: build a time-to-first-AE survival object (censored at
`censor_days`), fit a Kaplan-Meier curve, and read off the cumulative incidence (1 − survival)
and its 95% CI at 12 months.


In [ ]:
def ci_at_t(mrn_set, tox):
    sub_l1 = line1[line1['mrn'].isin(mrn_set)].copy()
    sub_b = batch_merged[batch_merged['mrn'].isin(mrn_set)].copy()
    if len(sub_l1) < 10:
        return 0.0, 0.0, 0.0
    ae_rec = sub_b[sub_b[tox] == 1].copy() if tox in sub_b.columns else pd.DataFrame()
    ae_rec = ae_rec[ae_rec['days_from_start'] >= 0] if len(ae_rec) > 0 else ae_rec
    if len(ae_rec) > 0:
        first_ae = ae_rec.groupby('mrn')['days_from_start'].min().reset_index().rename(
            columns={'days_from_start': 'time'})
        first_ae['event'] = 1
    else:
        first_ae = pd.DataFrame(columns=['mrn', 'time', 'event'])
    surv = sub_l1[['mrn', 'censor_days']].merge(first_ae, on='mrn', how='left')
    surv['event'] = surv['event'].fillna(0).astype(int)
    surv.loc[surv['event'] == 0, 'time'] = surv.loc[surv['event'] == 0, 'censor_days']
    surv.loc[(surv['event'] == 1) & (surv['time'] > surv['censor_days']), 'event'] = 0
    surv.loc[surv['event'] == 0, 'time'] = surv['censor_days']
    surv = surv[surv['time'] > 0].copy()
    if len(surv) < 10:
        return 0.0, 0.0, 0.0
    surv['time_months'] = surv['time'] / 30.44
    kmf = KaplanMeierFitter()
    kmf.fit(surv['time_months'], surv['event'])
    ci = (1 - kmf.survival_function_at_times(T_MONTHS).values[0]) * 100
    ci_tbl = kmf.confidence_interval_survival_function_
    idx = max(0, min(np.searchsorted(kmf.survival_function_.index, T_MONTHS, side='right') - 1,
                     len(ci_tbl) - 1))
    lo = (1 - ci_tbl.iloc[idx, 1]) * 100
    hi = (1 - ci_tbl.iloc[idx, 0]) * 100
    return ci, lo, hi


## Compute CI for every sex × toxicity, and export the values to CSV


In [ ]:
all_mrns = set(line1['mrn'])
records = []
group_mrns = {}

for grp in SEX_GROUPS:
    grp_mrns = set(sex_df.loc[sex_df['sex_clean'] == grp, 'mrn']) & all_mrns
    group_mrns[grp] = grp_mrns
    if len(grp_mrns) < 10:
        continue
    for tox in TOXICITY_COLUMNS:
        pct, lo, hi = ci_at_t(grp_mrns, tox)
        records.append({
            'sex': grp,
            'n_patients': len(grp_mrns),
            'toxicity': tox,
            'toxicity_display': TOXICITY_DISPLAY[tox],
            'cumulative_incidence_pct': round(pct, 3),
            'ci_95_lower': round(lo, 3),
            'ci_95_upper': round(hi, 3),
        })

results_df = pd.DataFrame(records)
results_df.to_csv(CSV_OUT, index=False)
print(f'Saved: {os.path.basename(CSV_OUT)}')
results_df


## Cox PH: Male vs. Female, per toxicity (Multivariable)

For each toxicity, fit a **multivariable** Cox proportional-hazards model on the pooled Male+Female
time-to-first-AE data. This matches the covariate adjustment used in the main figure panels (2C, 2D, 2E).

The model includes:
- `sex_female`: binary (0 = Male [reference], 1 = Female) — the exposure of interest
- `has_pd1_flag`, `has_ctla4_flag`: binary treatment flags (combined from raw columns)
- `contains_chemo`, `contains_hormone`, `contains_biologic`, `contains_targeted`: binary treatment flags
- `age_centered`: continuous, mean-centered age at LOT start
- `cancer_type_*`: dummy variables for cancer type (largest category as reference)

The Wald p-value on `sex_female` is reported as the actual number on the plot (not a
significance-star bucket), and the exponentiated coefficient is the hazard ratio.

**Cross-check:** a standard two-sample log-rank test (`p_value_logrank`) is computed on the exact
same survival data as an independent unadjusted check.

Per-arm N and event counts are also reported for each toxicity so low-power rows (e.g., a rare
event like hyperthyroidism) are visible directly in the table, not just inferable from a p-value.

Requires >=10 patients per arm and >=5 events pooled; otherwise returns NaN.


In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test

# ============================================================================
# Prepare covariate data for multivariable model
# Matches the adjustment covariates used in main figure panels (2C, 2D, 2E)
# ============================================================================

def _flag_on(x):
    """Convert various flag formats to boolean."""
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return x == 1
    return str(x).strip().lower() in ('1', 'true', 'yes')

# Build covariate dataframe with all adjustment variables
covar_cols = ['mrn', 'age_at_lot_start', 'cancer_type',
              'contains_ctla4_immuno', 'contains_ctla4', 'contains_non_ctla4_immuno', 'contains_pd1',
              'contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']
covar_df = patient_covars[[c for c in covar_cols if c in patient_covars.columns]].drop_duplicates('mrn').copy()

# Age
covar_df['age_at_lot_start'] = pd.to_numeric(covar_df['age_at_lot_start'], errors='coerce')

# Combined immuno flags (same logic as panel 2E)
covar_df['has_ctla4_flag'] = (covar_df['contains_ctla4_immuno'].apply(_flag_on) |
                              covar_df['contains_ctla4'].apply(_flag_on)).astype(int)
covar_df['has_pd1_flag'] = (covar_df['contains_non_ctla4_immuno'].apply(_flag_on) |
                            covar_df['contains_pd1'].apply(_flag_on)).astype(int)

# Treatment flags
for col in ['contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']:
    if col in covar_df.columns:
        covar_df[col] = covar_df[col].apply(_flag_on).astype(int)
    else:
        covar_df[col] = 0

# Clean cancer_type: fill missing with 'Unknown', standardize
covar_df['cancer_type'] = covar_df['cancer_type'].fillna('Unknown').astype(str).str.strip()
cancer_type_counts = covar_df['cancer_type'].value_counts()
CANCER_TYPE_REF = cancer_type_counts.index[0]  # Most common = reference

# Drop rows with missing required covariates
covar_df = covar_df[covar_df['age_at_lot_start'].notna()].copy()

ADJUST_COLS = ['has_pd1_flag', 'has_ctla4_flag', 'contains_chemo', 'contains_hormone',
               'contains_biologic', 'contains_targeted']
print(f"Adjustment covariates: {ADJUST_COLS} + age + cancer_type")
print(f"Cancer type reference (most common): {CANCER_TYPE_REF} (n={cancer_type_counts.iloc[0]:,})")
print(f"Cancer types: {len(cancer_type_counts)}")

def _build_surv(mrn_set, tox):
    """Time-to-first-AE survival table for the given mrn set and toxicity. Identical
    censoring/event logic to `ci_at_t` above -- centralized here so the KM curve, the Cox
    model, and the log-rank cross-check are all built from the exact same survival object.
    """
    sub_l1 = line1[line1['mrn'].isin(mrn_set)].copy()
    sub_b = batch_merged[batch_merged['mrn'].isin(mrn_set)].copy()
    if len(sub_l1) < 10:
        return None

    ae_rec = sub_b[sub_b[tox] == 1].copy() if tox in sub_b.columns else pd.DataFrame()
    ae_rec = ae_rec[ae_rec['days_from_start'] >= 0] if len(ae_rec) > 0 else ae_rec
    if len(ae_rec) > 0:
        first_ae = ae_rec.groupby('mrn')['days_from_start'].min().reset_index().rename(
            columns={'days_from_start': 'time'})
        first_ae['event'] = 1
    else:
        first_ae = pd.DataFrame(columns=['mrn', 'time', 'event'])

    surv = sub_l1[['mrn', 'censor_days']].merge(first_ae, on='mrn', how='left')
    surv['event'] = surv['event'].fillna(0).astype(int)
    surv.loc[surv['event'] == 0, 'time'] = surv.loc[surv['event'] == 0, 'censor_days']
    surv.loc[(surv['event'] == 1) & (surv['time'] > surv['censor_days']), 'event'] = 0
    surv.loc[surv['event'] == 0, 'time'] = surv['censor_days']
    surv = surv[surv['time'] > 0].copy()
    if len(surv) < 10:
        return None

    surv['time_months'] = surv['time'] / 30.44
    return surv


def cox_hr_pvalue_multivariable(ref_mrns, cmp_mrns, tox):
    """Multivariable Cox PH: comparison group (Female) vs reference group (Male), adjusting
    for treatment flags, age (centered), and cancer_type (dummy-encoded). Matches the covariate
    adjustment used in main figure panels. Returns (hazard_ratio, hr_lower_95, hr_upper_95,
    p_value, n_ref, n_cmp, events_ref, events_cmp); NaNs where underpowered or the fit fails.
    """
    n_ref_input, n_cmp_input = len(ref_mrns), len(cmp_mrns)
    if n_ref_input < 10 or n_cmp_input < 10:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan

    surv = _build_surv(set(ref_mrns) | set(cmp_mrns), tox)
    if surv is None:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan

    # Add sex indicator: 1 = Female (comparison), 0 = Male (reference)
    surv['sex_female'] = surv['mrn'].isin(cmp_mrns).astype(int)
    
    # Merge all covariates
    merge_cols = ['mrn', 'age_at_lot_start', 'cancer_type'] + ADJUST_COLS
    surv = surv.merge(covar_df[merge_cols], on='mrn', how='left')
    
    # Drop rows with missing required covariates
    surv = surv.dropna(subset=['age_at_lot_start', 'cancer_type'])
    
    if len(surv) < 20:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan
    
    n_ref = int((surv['sex_female'] == 0).sum())
    n_cmp = int((surv['sex_female'] == 1).sum())
    events_ref = int(surv.loc[surv['sex_female'] == 0, 'event'].sum())
    events_cmp = int(surv.loc[surv['sex_female'] == 1, 'event'].sum())

    if surv['event'].sum() < 5 or surv['sex_female'].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan, n_ref, n_cmp, events_ref, events_cmp

    # Center age for numerical stability
    surv['age_centered'] = surv['age_at_lot_start'] - surv['age_at_lot_start'].mean()
    
    # Dummy-encode cancer_type (drop most common as reference)
    cancer_dummies = pd.get_dummies(surv['cancer_type'], prefix='cancer', drop_first=False)
    ref_col = f'cancer_{CANCER_TYPE_REF}'
    if ref_col in cancer_dummies.columns:
        cancer_dummies = cancer_dummies.drop(columns=[ref_col])
    
    # Build model dataframe with all covariates
    # Order: exposure (sex), treatment flags, age, cancer_type dummies
    model_cols = ['time_months', 'event', 'sex_female'] + ADJUST_COLS + ['age_centered']
    model_df = pd.concat([surv[model_cols].reset_index(drop=True), 
                          cancer_dummies.reset_index(drop=True)], axis=1)
    
    # Fill any NaN in treatment flags with 0
    for col in ADJUST_COLS:
        model_df[col] = model_df[col].fillna(0).astype(int)
    
    # Fit multivariable Cox model with light ridge penalty for stability
    cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.0)
    try:
        cph.fit(model_df, duration_col='time_months', event_col='event')
    except Exception as e:
        print(f"  Cox fit failed for {tox}: {e}")
        return np.nan, np.nan, np.nan, np.nan, n_ref, n_cmp, events_ref, events_cmp

    hr = float(np.exp(cph.params_['sex_female']))
    hr_lower = float(cph.summary.loc['sex_female', 'exp(coef) lower 95%'])
    hr_upper = float(cph.summary.loc['sex_female', 'exp(coef) upper 95%'])
    p = float(cph.summary.loc['sex_female', 'p'])
    return hr, hr_lower, hr_upper, p, n_ref, n_cmp, events_ref, events_cmp


def logrank_pvalue(ref_mrns, cmp_mrns, tox):
    """Two-sample log-rank test on the same survival data, as an independent unadjusted
    cross-check. Note: log-rank is univariable, so it won't match the multivariable Cox
    p-value exactly, but provides a useful reference.
    """
    surv = _build_surv(set(ref_mrns) | set(cmp_mrns), tox)
    if surv is None:
        return np.nan
    is_cmp = surv['mrn'].isin(cmp_mrns)
    if is_cmp.sum() < 10 or (~is_cmp).sum() < 10:
        return np.nan
    try:
        lr = logrank_test(
            surv.loc[is_cmp, 'time_months'], surv.loc[~is_cmp, 'time_months'],
            event_observed_A=surv.loc[is_cmp, 'event'], event_observed_B=surv.loc[~is_cmp, 'event'],
        )
        return float(lr.p_value)
    except Exception:
        return np.nan


# Reference = first group listed in SEX_GROUPS ('Male'); compare each remaining group to it.
ref_group = SEX_GROUPS[0]
cmp_groups = SEX_GROUPS[1:]

cox_records = []
for tox in TOXICITY_COLUMNS:
    print(f"Fitting multivariable Cox for: {tox}")
    for cmp_grp in cmp_groups:
        hr, hr_lo, hr_hi, p, n_ref, n_cmp, ev_ref, ev_cmp = cox_hr_pvalue_multivariable(
            group_mrns[ref_group], group_mrns[cmp_grp], tox
        )
        lr_p = logrank_pvalue(group_mrns[ref_group], group_mrns[cmp_grp], tox)
        cox_records.append({
            'toxicity': tox,
            'toxicity_display': TOXICITY_DISPLAY[tox],
            'reference_group': ref_group,
            'comparison_group': cmp_grp,
            f'n_{ref_group.lower()}': n_ref,
            f'n_{cmp_grp.lower()}': n_cmp,
            f'events_{ref_group.lower()}': ev_ref,
            f'events_{cmp_grp.lower()}': ev_cmp,
            'hazard_ratio': round(hr, 3) if pd.notna(hr) else np.nan,
            'hr_lower_95': round(hr_lo, 3) if pd.notna(hr_lo) else np.nan,
            'hr_upper_95': round(hr_hi, 3) if pd.notna(hr_hi) else np.nan,
            'p_value_cox': p,
            'p_value_logrank_unadj': lr_p,
        })

cox_df = pd.DataFrame(cox_records)

# Merge HR / Cox p / log-rank p onto the comparison-group row of results_df (the reference-group
# row has nothing to compare itself to, so it gets NaN). One row per toxicity here since there
# are only 2 sex groups.
pmap = cox_df.set_index(['toxicity', 'comparison_group'])[['hazard_ratio', 'hr_lower_95', 'hr_upper_95', 'p_value_cox', 'p_value_logrank_unadj']]
results_df = results_df.merge(
    pmap, left_on=['toxicity', 'sex'], right_index=True, how='left'
)
results_df = results_df.rename(columns={
    'hazard_ratio': 'hr_vs_male_adj',
    'hr_lower_95': 'hr_vs_male_adj_lower_95',
    'hr_upper_95': 'hr_vs_male_adj_upper_95',
    'p_value_cox': 'p_value_vs_male_adj',
    'p_value_logrank_unadj': 'p_value_logrank_vs_male_unadj',
})

results_df.to_csv(CSV_OUT, index=False)
print(f'\nSaved (with multivariable Cox + unadjusted log-rank p-values): {os.path.basename(CSV_OUT)}')
print(f'Adjustments: age (centered), cancer_type (reference: {CANCER_TYPE_REF})')
cox_df


## What the p-value is adjusted for

**Covariate-adjusted (multivariable).** `p_value_vs_male_adj` is from a **multivariable** Cox model
that includes the same adjustment covariates used in the main figure panels (2C, 2D, 2E):

- Treatment: `has_pd1_flag`, `has_ctla4_flag`, `contains_chemo`, `contains_hormone`, `contains_biologic`, `contains_targeted`
- Demographics: age (continuous, centered), cancer type (dummy-encoded categorical)

The HR for sex represents the Female vs. Male hazard ratio **after adjusting for** confounding by
treatment regimen, age, and cancer type.

**Not adjusted for multiple comparisons, by default.** Six toxicities are each tested
independently, which inflates the false-positive rate across the family of six tests. The cell
below adds Benjamini-Hochberg FDR-adjusted and Bonferroni-adjusted p-values as extra columns
(`p_value_vs_male_adj_fdr_bh`, `p_value_vs_male_adj_bonferroni`) -- implemented directly with numpy (no
`statsmodels` dependency), verified to match `statsmodels.stats.multitest.multipletests` exactly.
Both the raw and multiplicity-corrected numbers land in the CSV. The figure itself still displays
the **covariate-adjusted** Cox p-value -- switch `PLOT_ADJUSTED_P` in the plot cell to `'fdr_bh'` or
`'bonferroni'` if the multiplicity-adjusted value should be shown on the brackets instead.


In [ ]:
def bh_fdr(pvals):
    """Benjamini-Hochberg FDR-adjusted p-values (q-values), NaN-safe. No external dependency --
    verified to match statsmodels.stats.multitest.multipletests(method='fdr_bh') exactly across
    ties, NaNs, and edge cases.
    """
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid_idx = np.where(~np.isnan(p))[0]
    m = len(valid_idx)
    if m == 0:
        return out
    vp = p[valid_idx]
    order = np.argsort(vp) # ascending p
    ranked_p = vp[order]
    ranks = np.arange(1, m + 1)
    q = ranked_p * m / ranks
    q = np.minimum.accumulate(q[::-1])[::-1]   # enforce monotonicity from the largest p down
    q = np.clip(q, 0, 1)
    out[valid_idx[order]] = q
    return out


def bonferroni_adjust(pvals):
    """Bonferroni-adjusted p-values, NaN-safe. Matches statsmodels exactly."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid_idx = np.where(~np.isnan(p))[0]
    m = len(valid_idx)
    if m == 0:
        return out
    out[valid_idx] = np.clip(p[valid_idx] * m, 0, 1)
    return out


# One test per toxicity (Female vs. Male); family size = 6 here.
raw_p = cox_df['p_value_cox'].values
cox_df['p_value_fdr_bh'] = bh_fdr(raw_p)
cox_df['p_value_bonferroni'] = bonferroni_adjust(raw_p)

# Merge the adjusted columns onto results_df / the CSV alongside the raw p-value.
adj_map = cox_df.set_index(['toxicity', 'comparison_group'])[['p_value_fdr_bh', 'p_value_bonferroni']]
results_df = results_df.merge(adj_map, left_on=['toxicity', 'sex'], right_index=True, how='left')
results_df = results_df.rename(columns={
    'p_value_fdr_bh': 'p_value_vs_male_adj_fdr_bh',
    'p_value_bonferroni': 'p_value_vs_male_adj_bonferroni',
})

results_df.to_csv(CSV_OUT, index=False)
print(f'Saved (with FDR/Bonferroni-adjusted p-values): {os.path.basename(CSV_OUT)}')
cox_df[['toxicity_display', 'p_value_cox', 'p_value_fdr_bh', 'p_value_bonferroni']]


## Plot

Grouped bars, retuned to the Nature 7pt / 6pt / 5pt text tiers. The Cox PH p-value (Female vs.
Male, `cox_df['p_value_cox']`) is shown as a bracket spanning each adverse-event's two bars, with
the p-value centered above it (`p < 0.001` below that threshold, exact value otherwise) --
standard journal-figure style. No `bbox_inches='tight'` on save -- place at 100% in Illustrator.


In [ ]:
# Which p-value column to draw on the brackets: 'cox' (raw, unadjusted -- default, matches
# the reference figure), 'fdr_bh', or 'bonferroni' (multiplicity-corrected across the 6 toxicities).
PLOT_ADJUSTED_P = 'cox'
P_COL = {'cox': 'p_value_cox', 'fdr_bh': 'p_value_fdr_bh', 'bonferroni': 'p_value_bonferroni'}[PLOT_ADJUSTED_P]

ae_list = TOXICITY_COLUMNS
display_names = [TOXICITY_DISPLAY[t] for t in ae_list]
groups = SEX_GROUPS
colors = SEX_GROUP_COLORS

def format_pval(p):
    """Real p-value, standard journal convention: exact value, or 'p < 0.001' once it's
    below that threshold (not a star/ns bucket).
    """
    if pd.isna(p):
        return ''
    if p < 0.001:
        return 'p < 0.001'
    return f'p = {p:.3f}'

n_ae = len(ae_list)
fig, ax = plt.subplots(figsize=(3.6, 2.3))
total_width = 0.55
bar_width = total_width / len(groups)
x = np.arange(n_ae)

for j, grp in enumerate(groups):
    grp_mrns = group_mrns[grp]
    if len(grp_mrns) < 10:
        continue
    sub = results_df[results_df['sex'] == grp].set_index('toxicity').loc[ae_list]
    g_pct = sub['cumulative_incidence_pct'].to_numpy()
    g_lo = sub['ci_95_lower'].to_numpy()
    g_hi = sub['ci_95_upper'].to_numpy()
    bar_off = -total_width / 2 + (j + 0.5) * bar_width
    ax.bar(x + bar_off, g_pct, bar_width, yerr=[g_pct - g_lo, g_hi - g_pct], capsize=1.5,
           color=colors[grp], edgecolor='none', label=f'{grp} (N={len(grp_mrns):,})',
           ecolor='gray', error_kw={'linewidth': 0.5})

ax.set_xlabel('Adverse event', fontsize=7)
ax.set_ylabel('1-year cumulative\nincidence (%, 95% CI)', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(display_names, rotation=30, ha='right', fontsize=6)
ax.tick_params(axis='y', labelsize=6)
ax.legend(fontsize=5, framealpha=0.95, loc='upper right', handlelength=1)
ax.set_ylim(0, min(ax.get_ylim()[1] * 1.32, 60))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ---- p-value brackets (Female vs. Male, per toxicity) ----
p_lookup = cox_df.set_index('toxicity')[P_COL]
y_range = ax.get_ylim()[1]
tick_h = y_range * 0.025
gap = y_range * 0.05
for i, tox in enumerate(ae_list):
    if tox not in p_lookup.index:
        continue
    label = format_pval(p_lookup.loc[tox])
    if not label:
        continue
    xs, tops = [], []
    for j, grp in enumerate(groups):
        if len(group_mrns[grp]) < 10:
            continue
        bar_off = -total_width / 2 + (j + 0.5) * bar_width
        xs.append(x[i] + bar_off)
        row = results_df[(results_df['sex'] == grp) & (results_df['toxicity'] == tox)]
        if len(row):
            tops.append(row['ci_95_upper'].values[0])
    if len(xs) < 2 or not tops:
        continue
    x_left, x_right = min(xs), max(xs)
    y_bar = max(tops) + gap
    y_tick = y_bar - tick_h
    ax.plot([x_left, x_left, x_right, x_right], [y_tick, y_bar, y_bar, y_tick],
            color='black', linewidth=0.6)
    ax.text((x_left + x_right) / 2, y_bar + gap * 0.3, label, ha='center', va='bottom', fontsize=5)

plt.tight_layout()

plt.show()

In [ ]:
with PdfPages(PDF_OUT) as pdf:
    pdf.savefig(fig, dpi=450)
plt.close(fig)
print(f'Saved: {os.path.basename(PDF_OUT)}')

## Sanity check: figure p-values match the CSV

Re-reads `CSV_OUT` from disk (independent of the in-memory `cox_df`/`results_df` used to draw
the plot) and compares, per toxicity, the p-value on the figure brackets to the p-value in the
saved CSV. Also recomputes the exact bracket label (`format_pval`) from both sources so a
formatting bug (not just a value bug) would be caught too.


In [ ]:
csv_on_disk = pd.read_csv(CSV_OUT)

check_records = []
for _, r in cox_df.iterrows():
    tox, cmp_grp = r['toxicity'], r['comparison_group']
    p_figure = r[P_COL]

    csv_col = {'p_value_cox': 'p_value_vs_male_adj',
               'p_value_fdr_bh': 'p_value_vs_male_adj_fdr_bh',
               'p_value_bonferroni': 'p_value_vs_male_adj_bonferroni'}[P_COL]
    row = csv_on_disk[(csv_on_disk['toxicity'] == tox) & (csv_on_disk['sex'] == cmp_grp)]
    p_csv = row[csv_col].values[0] if len(row) else np.nan

    check_records.append({
        'toxicity': tox,
        'p_value_source_column': csv_col,
        'p_on_figure': p_figure,
        'p_in_csv': p_csv,
        'label_on_figure': format_pval(p_figure),
        'label_from_csv': format_pval(p_csv),
        'values_match': bool(np.isclose(p_figure, p_csv, equal_nan=True)),
        'labels_match': format_pval(p_figure) == format_pval(p_csv),
    })

consistency_df = pd.DataFrame(check_records)
all_ok = bool(consistency_df['values_match'].all() and consistency_df['labels_match'].all())
print(f'Figure p-values consistent with CSV ("{P_COL}"): {all_ok}')
assert all_ok, 'Figure and CSV p-values disagree -- see consistency_df for the mismatched row(s).'
consistency_df